# Morris-Lecar: one- and two-parameter diagrams

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/16_morris_lecar_bifurcation.ipynb)

The workflow of XPP Studio's figure script, done from Python: the phase plane at
`i = 60`, the equilibria in `i` with the periodic orbits from the first Hopf point, and
that Hopf point followed in two parameters (`i`, `phi`).

*Based on an example from [XPP Studio](https://xpp-studio.github.io/xpp-studio/) (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.4 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.4.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("morris_lecar_figures")
      .function("minf(v)", ".5*(1+tanh((v-v1)/v2))")
      .function("winf(v)", ".5*(1+tanh((v-v3)/v4))")
      .function("tauw(v)", "1/cosh((v-v3)/(2*v4))")
      .ode("v", "(i+gl*(vl-v)+gk*w*(vk-v)+gca*minf(v)*(vca-v))/c")
      .ode("w", "phi*(winf(v)-w)/tauw(v)")
      .aux("ica", "gca*minf(v)*(v-vca)")
      .parameter(i=60, c=20, vk=-84, vl=-60, vca=120, gk=8, gl=2, gca=4.4)
      .parameter(v1=-1.2, v2=18, v3=2, v4=30, phi=0.04)
      .initial(v=-60, w=0.015)
      .option(total=400, dt=0.25, meth="rk4", bound=10000))
print(mb.render())

minf(v)=.5*(1+tanh((v-v1)/v2))
winf(v)=.5*(1+tanh((v-v3)/v4))
tauw(v)=1/cosh((v-v3)/(2*v4))
v'=(i+gl*(vl-v)+gk*w*(vk-v)+gca*minf(v)*(vca-v))/c
w'=phi*(winf(v)-w)/tauw(v)
aux ica=gca*minf(v)*(v-vca)
par i=60,c=20,vk=-84,vl=-60,vca=120,gk=8,gl=2,gca=4.4
par v1=-1.2,v2=18,v3=2,v4=30,phi=0.04
init v=-60,w=0.015
@ total=400,dt=0.25,meth=rungekutta,bound=10000
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="v", y="w", window=(-75, 45, -0.1, 0.55),
              sliders={"i": (0, 300), "phi": (0.01, 0.4)},
              auto="i", auto_range=(0, 300), auto2="phi", auto2_range=(0, 0.4), title="Morris-Lecar figures")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**The three figures in one go.** Equilibria in `i`, orbits from HB1 (with l1), and HB1
in (`i`, `phi`) with any GH points.

In [5]:
parts = ex.run_auto()
hb1 = min((p for d in parts for p in d.special("HB")), key=lambda p: p.x)
ex.pick("HB", hb1.x)
po, l1 = ex.periodic()
curves, gh = ex.two_parameter()
print(f"HB1 at i = {hb1.x:.4f}, l1 = {l1:.4g}; Hopf curve {sum(len(c) for c in curves)} points; GH: {[(round(g['par'],3), round(g['par2'],4)) for g in gh]}")

HB1 at i = 93.8575, l1 = 0.006543; Hopf curve 4000 points; GH: []


## Things to try

- Look at the two-parameter plot: where along the Hopf curve does the onset change from sub- to supercritical (GH)?